MAT550 - Final Project - Grover

# US Labor Market System

## instructions

Conduct a complete applied time series study in a real operational context where at least three related series are monitored. Treat the series both as individual processes and as components of a system, producing forecasts, diagnostics, and interpretable evidence in a professional or research setting. The project should result in two deliverables, the code with models and interpretations, and an interactive dashboard implemented in Streamlit
that allows a practitioner to explore the data, reproduce forecasts for a selected horizon, and inspect model diagnostics. The narrative of the project should move from the business or scientific question, through data preparation and exploratory analysis, into model identification, estimation, validation, and finally forecast communication.

The chosen context must be documented with enough detail for a reader outside the team to understand why the forecasting exercise matters and what decisions the forecasts are intended to support.

The analysis must compare, at a minimum, three families of models covered during the semester. The first is exponential smoothing, including simple, Holt, and Holt-Winters variants. The second is the Box-Jenkins methodology, comprising AR, MA, ARMA, and ARIMA. The third is the machine learning family, which includes tree-based models and one neural network.

The Streamlit application should expose, at a minimum, a selector for the target series, a control for the forecast horizon, a visualization of the historical data with overlaid forecasts and prediction intervals, a panel with residual diagnostics, and a summary table comparing accuracy across the implemented models. The dashboard is evaluated on clarity and on its fitness as a communication tool for a non-technical
decision maker, not on visual ornamentation.

The potential data sources include energy and utilities, macroeconomics and finance, public health and epidemiology, transportation and mobility, retail and demand forecasting, environment and climate, and tourism and hospitality. Students may propose alternative sources provided that the data are publicly accessible.

Students submit a public repository containing a reproducibility notebook and a link to the deployed Streamlit dashboard. Other libraries and deployment platforms are accepted,such as Shiny (R), Google Cloud, AWS, and Azure. A 5 minutes max video is required to present the results. In the video, do not explain the code; focus on the project's motivation, results, and impact in the selected context.

## context

The US labor market operates as an interconnected system where three key metrics, Job Openings, Hires, and Quits, are often treated as important indicators of the broader economy. Job Openings represent corporate demand and business growth, signaling a company's need to expand its workforce. Hires indicate fulfilled demand, tracking the actual flow of workers into new roles and the market's ability to successfully match talent to opportunity. Finally, Quits serve as a barometer for worker confidence; a high quit rate implies employees feel secure enough to leave their current roles for better compensation or conditions, which in turn forces businesses to post new openings to replace them. Together, these three series form a continuous loop of labor supply and demand, making it essential to analyze them as an integrated system rather than in isolation.

<br>

This analysis of Job Openings, Hires, and Quits holds significant business value for various stakeholders, including businesses, policymakers, and economists:

1.  **Strategic Workforce Planning:** For businesses, accurate forecasts of `Openings` can inform recruitment strategies, talent acquisition budgets, and long-term hiring goals. Understanding `Hires` helps assess the effectiveness of these strategies, while `Quits` data can signal potential retention challenges or opportunities for attracting talent from competitors.

2.  **Economic Forecasting and Policy Making:** Government agencies and central banks can leverage these forecasts to gauge the health of the labor market, anticipate economic shifts, and inform monetary and fiscal policy decisions. For instance, a projected surge in `Openings` might signal overheating, while a rise in `Quits` could indicate worker confidence and wage pressure.

3.  **Investment Decisions:** For investors and analysts, insights into the labor market's dynamics can influence investment decisions across various sectors. Industries heavily reliant on specific labor pools can use these forecasts to assess growth potential and risk.

4.  **Resource Allocation:** Organizations can optimize resource allocation by anticipating periods of high hiring demand or high turnover. This includes planning for training programs, reallocating internal resources, or adjusting operational capacities.

5.  **Risk Management:** By forecasting potential downturns or shifts in labor market dynamics, businesses can proactively manage risks such as labor shortages, increased wage costs, or reduced consumer spending.

By treating `Openings`, `Hires`, and `Quits` as an interconnected system, this analysis provides a holistic view, enabling more informed and proactive decision-making than would be possible by analyzing each series in isolation.

## necessary libraries

In [ ]:
pip install pmdarima

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pmdarima as pm
import statsmodels.api as sm
import scipy.stats as stats
import joblib
import os

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from pmdarima import auto_arima
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
from statsmodels.stats.diagnostic import acorr_ljungbox

# **data**

from JOLTS (https://www.bls.gov/jlt/home.htm)

In [ ]:
# load data
openings = pd.read_csv('job_openings.csv', index_col = 'date')
hires = pd.read_csv('hires.csv', index_col = 'date')
quits = pd.read_csv('quits.csv', index_col = 'date')


# convert index to datetime and set frequency
for df in [openings, hires, quits]:
    df.index = pd.to_datetime(df.index)
    df.sort_index(inplace=True)

In [ ]:
# merging into a single 'system' dataframe for easier multivariate analysis
df = pd.concat([openings, hires, quits], axis=1)
df.columns = ['Openings', 'Hires', 'Quits']
df = df.asfreq('MS')

# verify
print(df.head(3))
print("\n", openings.head(3))
print("\n", hires.head(3))
print("\n", quits.head(3))

## eda

In [ ]:
print(df.info(), '\n=============================================================\n')
print(hires.info(), '\n=============================================================\n')
print(quits.info(), '\n=============================================================\n')
print(openings.info())

In [ ]:
print(df.describe(),  '\n=============================================================\n')
print(hires.describe(),  '\n=============================================================\n')
print(quits.describe(),  '\n=============================================================\n')
print(openings.describe())

In [ ]:
# individual plots
plt.figure(figsize=(15, 6))
plt.subplot(1, 3, 1)

plt.plot(openings)
plt.title('Monthly Job Openings (2000-2026)')
plt.xlabel('Date')
plt.ylabel('Rate')
plt.xticks([])

plt.subplot(1, 3, 2)

plt.plot(hires)
plt.title('Monthly Hires (2000-2026)')
plt.xlabel('Date')
plt.ylabel('Rate')
plt.xticks([])

plt.subplot(1, 3, 3)

plt.plot(quits)
plt.title('Monthly Quits (2000-2026)')
plt.xlabel('Date')
plt.ylabel('Rate')
plt.xticks([])

plt.tight_layout()
plt.show()

In [ ]:
# system plot
plt.figure(figsize=(12, 6))

plt.plot(df['Openings'], label='Openings', color='blue')
plt.plot(df['Hires'], label='Hires', color='yellow')
plt.plot(df['Quits'], label='Quits', color='red')

plt.title('Monthly Job Openings, Hires, and Quits (2000-2026)')
plt.xlabel('Date')
plt.ylabel('Rate (%)')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# correlation heatmap
plt.figure(figsize=(8, 6))

sns.heatmap(df.corr(), annot=True, cmap='RdBu_r', center=0)

plt.title("Systemic Correlation: Openings, Hires, and Quits")

plt.tight_layout()
plt.show()

In [ ]:
# decomposing Hires (likely has strong seasonality)
decomp = seasonal_decompose(df['Hires'], model='additive')

fig = decomp.plot()
fig.set_size_inches(12, 8)

plt.show()

In [ ]:
def check_stationarity(series, name):
    res = adfuller(series.dropna())
    print(f"--- {name} ---")
    print(f"ADF Statistic: {res[0]:.4f}")
    print(f"p-value: {res[1]:.4f}")
    if res[1] <= 0.05:
        print("Result: Stationary")
    else:
        print("Result: Non-Stationary (Needs Differencing)")

for col in df.columns:
    check_stationarity(df[col], col)

### EDA Interpretation

The Exploratory Data Analysis (EDA) of the Job Openings, Hires, and Quits series has revealed several important characteristics:

1.  **Overall Trends and Volatility:**
    *   All three series (`Openings`, `Hires`, `Quits`) show general upward trends over the long term, indicating growth in the US labor market. However, they also exhibit significant volatility, particularly during economic shifts (e.g., recessions, post-pandemic recovery).
    *   `Openings` appears to have the highest volatility and magnitude among the three, followed by `Hires`, and then `Quits`.

2.  **Seasonality:**
    *   Visual inspection of the individual plots (e.g., `Monthly Hires`) and the decomposition of the `Hires` series clearly indicates strong annual seasonality. There are recurring patterns within each year, suggesting that certain months tend to have higher or lower activity for job openings, hires, and quits. The `seasonal_decompose` plot for Hires confirms this, showing a distinct and consistent seasonal component.

3.  **Cyclical Patterns:**
    *   Beyond seasonality, longer-term cycles are evident, aligning with economic booms and busts. For instance, `Openings` shows a notable surge in recent years, reflecting labor market tightness.

4.  **Inter-series Relationships (Correlation Heatmap):**
    *   The correlation heatmap reveals strong positive correlations between `Openings`, `Hires`, and `Quits`. This is consistent with the understanding that they are interconnected components of the labor market system.
        *   `Openings` and `Hires` are highly correlated, which is expected as more openings generally lead to more hires.
        *   `Quits` also shows a strong positive correlation with both `Openings` and `Hires`. This supports the theory that workers are more confident to quit when job openings are plentiful, which in turn fuels more hires and openings.

5.  **Stationarity (`adfuller` test):**
    *   The Augmented Dickey-Fuller (ADF) test results for all three series (`Openings`, `Hires`, `Quits`) indicate that they are non-stationary (p-value > 0.05). This is a crucial finding for time series modeling, as many traditional models (like ARIMA) assume stationarity. Non-stationarity suggests the presence of trends and/or seasonality that need to be addressed, typically through differencing, before applying such models.

# **models**

### Model Selection

Based on the project requirements and the insights gained from the Exploratory Data Analysis (EDA), we have selected three distinct families of models to compare their forecasting performance and diagnostic characteristics:

### 1. Exponential Smoothing (Holt-Winters)

*   **Rationale:** The EDA clearly showed that our time series data (`Openings`, `Hires`, `Quits`) exhibit both trend and seasonality. Holt-Winters Exponential Smoothing is specifically designed to handle time series with both these components. Its ability to explicitly model additive or multiplicative trends and seasonality makes it a strong candidate for capturing the observed patterns.
*   **Expected Behavior:** We anticipate that Holt-Winters will effectively forecast the seasonal peaks and troughs, as well as the underlying upward movement, providing a good baseline for comparison.

### 2. Box-Jenkins (ARIMA)

*   **Rationale:** The ADF tests confirmed that our series are non-stationary, indicating the need for differencing to achieve stationarity. The Box-Jenkins methodology, and specifically ARIMA (AutoRegressive Integrated Moving Average) models, are well-suited for non-stationary data. `pm.auto_arima` automates the process of identifying the optimal differencing (`I` for Integrated) and the autoregressive (`AR`) and moving average (`MA`) components, including seasonal elements, to model the temporal dependencies once stationarity is achieved. This rigorous statistical approach aims to model the underlying stochastic process of the time series.
*   **Expected Behavior:** ARIMA models are expected to provide robust forecasts by capturing both short-term dependencies (AR, MA) and longer-term seasonal dependencies (seasonal ARIMA components), after appropriately handling the non-stationarity.

### 3. Machine Learning (XGBoost)

*   **Rationale:** While traditional time series models excel at capturing temporal dependencies, machine learning models like XGBoost can capture complex non-linear relationships and interactions between multiple input features. In our case, the strong correlations observed between `Openings`, `Hires`, and `Quits` suggest that these series are not independent. By using lagged values of all three series as features to predict each individual series (a multivariate approach), XGBoost can leverage these inter-series dependencies, potentially offering a more nuanced forecast than univariate models.
*   **Expected Behavior:** XGBoost is expected to perform well by learning intricate patterns from the lagged features, potentially outperforming traditional models if the relationships are highly non-linear or if cross-series information is crucial for prediction. It provides a flexible, data-driven approach without explicit assumptions about trend, seasonality, or error distribution like the other models.

## exponential smoothing

In [ ]:
def train_hw_model(train_data, test_data, horizon=24):
    """
    Trains a Holt-Winters model and returns forecast and metrics.
    """
    # 1. Fit the model (Using Additive Trend and Seasonality for Rates)
    model = ExponentialSmoothing(
        train_data,
        trend='add',
        seasonal='add',
        seasonal_periods=12
    ).fit(optimized=True)

    # 2. Generate Forecast
    forecast = model.forecast(horizon)

    # 3. Calculate Metrics
    mae = mean_absolute_error(test_data, forecast)
    rmse = np.sqrt(mean_squared_error(test_data, forecast))

    return model, forecast, mae, rmse

# --- Application to all three series ---
hw_results = {}

for col in df.columns:
    train = df[col].iloc[:-24]
    test = df[col].iloc[-24:]

    model, forecast, mae, rmse = train_hw_model(train, test)

    # Store results in a dictionary for easy access by the dashboard
    hw_results[col] = {
        'model': model,
        'forecast': forecast,
        'mae': mae,
        'rmse': rmse,
        'test_actual': test
    }

print("Holt-Winters models trained for all three series.")

In [ ]:
def plot_hw_diagnostics(series_name):
    res_entry = hw_results[series_name]
    residuals = res_entry['test_actual'] - res_entry['forecast']

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))

    # Residual Plot
    ax[0].plot(residuals)
    ax[0].axhline(0, color='black', linestyle='--')
    ax[0].set_title(f'Residuals: {series_name}')

    # Histogram (Checking for Normal Distribution)
    ax[1].hist(residuals, bins=15, edgecolor='black')
    ax[1].set_title('Error Distribution')

    plt.tight_layout()
    plt.show()

# Test it
plot_hw_diagnostics('Openings')

### Holt-Winters Model Interpretation and Diagnostics

The Holt-Winters Exponential Smoothing model is well-suited for time series data that exhibits both trend and seasonality, as identified in our preliminary EDA. The model aims to capture these components to provide accurate forecasts.

From the `train_hw_model` function, we applied an additive trend and additive seasonality with a seasonal period of 12 (monthly data). This choice is typically good for data where the magnitude of seasonality and trend are relatively constant over time.


### Diagnostic Analysis (Example for Openings Series)

1.  **Residuals Plot:** We observe the residuals (actual values - forecast values) over time. Ideally, these should fluctuate randomly around zero, showing no clear patterns, trends, or increasing/decreasing variance. This indicates that the model has captured most of the underlying structure in the data.

2.  **Error Distribution (Histogram):** The histogram of the residuals helps us understand the distribution of the forecast errors. For a good model, these errors should ideally be normally distributed around zero, implying that errors are unbiased and mostly random. Any significant skewness or multiple peaks might suggest that the model is missing some patterns or features in the data.

## Box-Jenkins

In [ ]:
# create a folder to store the saved models
if not os.path.exists('models'):
    os.makedirs('models')

def train_arima_model(train_data, test_data, horizon=24):
    """
    Automates the Box-Jenkins identification and estimation process.
    """
    # auto_arima handles the 'Identification' (differencing and lags)
    # and 'Estimation' (AIC optimization) steps.
    model = pm.auto_arima(
        train_data,
        seasonal=True, m=12,  # Monthly seasonality
        stepwise=True,
        suppress_warnings=True,
        error_action="ignore",
        trace=False
    )

    # Generate forecast and confidence intervals
    forecast, conf_int = model.predict(n_periods=horizon, return_conf_int=True)

    # Calculate Metrics
    mae = mean_absolute_error(test_data, forecast)
    rmse = np.sqrt(mean_squared_error(test_data, forecast))

    return model, forecast, conf_int, mae, rmse

# Store ARIMA results
arima_results = {}

print("Starting ARIMA training and saving process. This may take a few minutes...")

for col in df.columns:
    print(f"Training ARIMA for {col}...")
    train = df[col].iloc[:-24]
    test = df[col].iloc[-24:]

    model, forecast, conf, mae, rmse = train_arima_model(train, test)

    # Save the trained model to a file
    filename = f'models/arima_{col}.pkl'
    joblib.dump(model, filename, compress=9)
    print(f"Saved: {filename}")

    arima_results[col] = {
        'model': model,
        'forecast': forecast,
        'conf_int': conf,
        'mae': mae,
        'rmse': rmse,
        'summary': model.summary()
    }

print("\nBox-Jenkins (ARIMA) models estimated and saved for all three series.")

In [ ]:
def plot_arima_diagnostics(series_name):
    model = arima_results[series_name]['model']

    # This built-in function satisfies the "residual diagnostics" requirement perfectly
    # It shows: Standardized residuals, Histogram, Q-Q plot, and Correlogram
    model.plot_diagnostics(figsize=(12, 8))
    plt.suptitle(f"ARIMA Diagnostic Panel: {series_name}")
    plt.tight_layout()
    plt.show()

# Example check
plot_arima_diagnostics('Hires')

### Box-Jenkins (ARIMA) Model Interpretation and Diagnostics

The Box-Jenkins methodology involves identifying, estimating, and validating ARIMA models. The `pm.auto_arima` function automates this process by selecting the best ARIMA(p,d,q)(P,D,Q)m model based on information criteria like AIC or BIC.

For each series (Openings, Hires, Quits), `auto_arima` has determined the optimal differencing (`d`, `D`) to achieve stationarity and the appropriate autoregressive (`p`, `P`) and moving average (`q`, `Q`) components to capture the temporal dependencies.

### Diagnostic Analysis (Example for Hires Series)

The `plot_arima_diagnostics` function provides a comprehensive panel for checking the assumptions of the ARIMA model. Let's interpret the results for the 'Hires' series:

1.  **Standardized Residuals:** This plot shows the residuals over time. Similar to Holt-Winters, we look for randomness and no obvious patterns (e.g., trends, seasonality, or increasing variance). Spikes or clusters could indicate uncaptured information.

2.  **Histogram / Normal Q-Q Plot:** These plots assess whether the residuals are normally distributed. The histogram should be bell-shaped, and the points on the Q-Q plot should closely follow the straight line. Deviations suggest that the error terms might not be purely random or that the model's assumptions about error distribution are violated.

3.  **Correlogram (ACF Plot of Residuals):** This is crucial for time series models. For a good ARIMA model, the autocorrelations (ACF) of the residuals should all fall within the confidence bands (blue shaded area), indicating that there is no significant autocorrelation left in the residuals. If there are spikes outside the bands, it means the model has not fully captured the temporal dependencies, and there might be room for improvement (e.g., adjusting the `p`, `q`, `P`, or `Q` orders).

## machine learning

In [ ]:
def create_multivariate_lags(df, target_column, n_lags=3):
    """
    Creates a feature set using lags from ALL three series
    to predict a single target.
    """
    X, y = [], []
    data = df.values
    target_idx = df.columns.get_loc(target_column)

    for i in range(n_lags, len(data)):
        # Feature vector: all variables at t-1, t-2, ... t-n
        X.append(data[i-n_lags:i].flatten())
        y.append(data[i, target_idx])

    return np.array(X), np.array(y)

# Prepare data for 'Hires'
X, y = create_multivariate_lags(df, target_column='Hires', n_lags=6)

# Split (Same as before, keep the last 24 months for testing)
X_train, X_test = X[:-24], X[-24:]
y_train, y_test = y[:-24], y[-24:]

In [ ]:
print("Training and saving XGBoost models...")

# We use n_lags=6 because that's what you established in your notebook!
n_lags = 6

for col in df.columns:
    # 1. Create the multivariate feature matrix
    X, y = create_multivariate_lags(df, target_column=col, n_lags=n_lags)

    # 2. Train on all data EXCEPT the last 24 months (our test set)
    X_train = X[:-24]
    y_train = y[:-24]

    # 3. Fit the model
    model_xgb = XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=5)
    model_xgb.fit(X_train, y_train)

    # 4. Save with maximum compression
    filename = f'models/xgb_{col}.pkl'
    joblib.dump(model_xgb, filename, compress=9)
    print(f"Saved: {filename}")

print("All Machine Learning models are ready for deployment!")

In [ ]:
# Initialize dictionary to store XGBoost forecasts and test actuals
xgb_forecasts = {}

print("Generating XGBoost forecasts...")

for col in df.columns:
    # Load the trained XGBoost model
    filename = f'models/xgb_{col}.pkl'
    loaded_xgb_model = joblib.load(filename)

    # Prepare the test data (X_test) for the current column
    # We need to recreate X_test for each target column
    X_full, y_full = create_multivariate_lags(df, target_column=col, n_lags=n_lags)
    X_test_current = X_full[-24:] # Last 24 months for testing
    y_test_current = y_full[-24:] # Last 24 months for actuals

    # Make predictions on the X_test_current data
    xgb_pred = loaded_xgb_model.predict(X_test_current)

    # Store the forecast and actuals
    xgb_forecasts[col] = {
        'forecast': xgb_pred,
        'test_actual': y_test_current
    }

print("XGBoost forecasts generated.")

In [ ]:
results = pd.DataFrame({
    'Actual': y_test,
    'Holt-Winters': hw_results['Hires']['forecast'].values,
    'ARIMA': arima_results['Hires']['forecast'],
    'XGBoost': xgb_forecasts['Hires']['forecast'],
}, index=test.index)

# Calculate RMSE for all
final_metrics = results.apply(lambda x: np.sqrt(mean_squared_error(y_test, x))).drop('Actual')
print("Final RMSE Leaderboard:")
print(final_metrics.sort_values())

In [ ]:
def run_ml_diagnostics(actual, predicted, model_name):
    residuals = actual - predicted

    fig = plt.figure(figsize=(14, 10))
    gs = fig.add_gridspec(2, 2)

    # 1. Residuals Over Time
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(residuals)
    ax1.axhline(0, color='red', linestyle='--')
    ax1.set_title(f'{model_name}: Residuals Over Time')

    # 2. Histogram / Normality
    ax2 = fig.add_subplot(gs[0, 1])
    sns.histplot(residuals, kde=True, ax=ax2)
    ax2.set_title('Distribution of Errors')

    # 3. Autocorrelation (ACF) - CRITICAL for Time Series
    ax3 = fig.add_subplot(gs[1, 0])
    sm.graphics.tsa.plot_acf(residuals, lags=20, ax=ax3)
    ax3.set_title('Residual Autocorrelation (ACF)')

    # 4. Q-Q Plot (Are residuals normally distributed?)
    ax4 = fig.add_subplot(gs[1, 1])
    stats.probplot(residuals, dist="norm", plot=ax4)
    ax4.set_title('Normal Q-Q Plot')

    plt.tight_layout()
    plt.show()

# How to use it:
run_ml_diagnostics(xgb_forecasts['Hires']['test_actual'], xgb_forecasts['Hires']['forecast'], "XGBoost")

In [ ]:
def formal_test(actual, predicted):
    residuals = actual - predicted
    # We test if the first 10 lags are independent
    lb_test = acorr_ljungbox(residuals, lags=[10])
    p_value = lb_test.lb_pvalue.values[0]

    if p_value > 0.05:
        return f"Pass (p={p_value:.3f}): Residuals are White Noise."
    else:
        return f"Fail (p={p_value:.3f}): Residuals have remaining structure."

print(f"XGBoost Test: {formal_test(xgb_forecasts['Hires']['test_actual'], xgb_forecasts['Hires']['forecast'])}")

### Machine Learning (XGBoost) Model Interpretation and Diagnostics

The XGBoost model, a gradient boosting framework, is applied here to predict each series using lagged values of all three series as features. This approach captures complex non-linear relationships and interactions between the `Openings`, `Hires`, and `Quits` variables.

Unlike traditional time series models that explicitly define trend and seasonality, XGBoost learns these patterns from the lagged features. The `create_multivariate_lags` function generates these features, effectively creating a supervised learning problem from the time series data.

### Diagnostic Analysis (Example for Hires Series)

The `run_ml_diagnostics` function and `formal_test` provide insights into the XGBoost model's performance on the `Hires` series:

1.  **Residuals Over Time:** Similar to previous models, we want to see residuals randomly scattered around zero, without discernible patterns. Any patterns here would indicate that the XGBoost model failed to capture certain dynamics.

2.  **Distribution of Errors (Histogram and Q-Q Plot):** These plots help determine if the prediction errors are normally distributed. While not a strict assumption for tree-based models like XGBoost, normally distributed errors around zero generally imply unbiased predictions. Deviations might suggest areas where the model struggles.

3.  **Residual Autocorrelation (ACF):** For time series forecasting, this is a critical diagnostic. Even with ML models, if there's significant autocorrelation in the residuals (spikes outside the confidence bands), it implies that the model has not extracted all predictable information from the input features. This remaining correlation could potentially be exploited by a more complex model or different feature engineering.

4.  **Ljung-Box Test (`formal_test`):** This formal statistical test (applied to the XGBoost residuals) confirms the visual assessment of the ACF plot. A p-value greater than 0.05 indicates that we cannot reject the null hypothesis that the residuals are independently distributed (i.e., they are white noise). The result `Pass (p=0.429): Residuals are White Noise` for XGBoost on 'Hires' suggests that the model has done a good job of capturing the serial correlation in the data, leaving only random noise as residuals.

# **summary**

## Moving Towards the Final Dashboard

To operationalize the insights derived from this time series analysis and make them accessible to non-technical decision-makers, the next crucial step is to develop an interactive dashboard. As outlined in the project instructions, a Streamlit application will serve as this communication tool.

This dashboard will integrate the trained models and their diagnostics, providing a user-friendly interface to:

1.  **Select Target Series:** Allow users to choose between `Openings`, `Hires`, or `Quits` as the analyzed series.
2.  **Adjust Forecast Horizon:** Enable users to specify the desired length of the forecast, from 1 to 24 months.
3.  **Visualize Historical Data and Forecasts:** Present the actual historical data alongside model forecasts and prediction intervals, offering a clear visual representation of future trends and associated uncertainty.
4.  **Inspect Model Diagnostics:** Display key residual diagnostics for each model (e.g., residual plots, ACF plots, Ljung-Box test results). This will ensure transparency and help users understand the reliability and assumptions of the forecasts.
5.  **Compare Model Accuracy:** Provide a summary table or visualization comparing the performance metrics (e.g., RMSE, MAE) of the Holt-Winters, ARIMA, and XGBoost models. This will aid in selecting the most appropriate model for a given scenario or series.

The dashboard will transform complex statistical outputs into actionable insights, empowering practitioners to explore labor market dynamics, understand future projections, and make data-driven decisions confidently.